# Preparing the notebook environment

In [ ]:
# %pip install transformers librosa torch torchaudio numpy tqdm
# %pip install ipywidgets

# Extract and Save embeddings - code

## 1. HuBERT and 2. WavLM

In [1]:
import os
import gc
import torch
import librosa
import numpy as np
# from tqdm.notebook import tqdm
from tqdm import tqdm
from transformers import AutoFeatureExtractor, AutoModel

# --- Configuration ---
DATA_DIR = "curated_dataset/eval" # Update if your Kaggle path differs (e.g., /kaggle/input/...)
OUTPUT_DIR = "model_embeddings_eval"
SR = 16000
CHUNK_LENGTH_S = 5
CHUNK_SAMPLES = SR * CHUNK_LENGTH_S

# Map folders to integer labels
CLASSES = sorted(os.listdir(DATA_DIR))
LABEL_MAP = {cls_name: idx for idx, cls_name in enumerate(CLASSES)}

# Hugging Face Model Repositories
MODELS = {
    "HuBERT": "facebook/hubert-base-ls960",
    "WavLM": "microsoft/wavlm-base-plus",
    # other models are not available directly in Hugging face
    # "AVES": "EarthSpeciesProject/esp-aves2-sl-beats-bio", # Update if using a different fork
    # "BIRDAves": "DBD-research-group/BirdAVES",    
    # "MusicHuBERT": "m-a-p/MERT-v1-95M", 
    # "XEUS": "espnet/xeus"
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def process_audio(file_path):
    """Loads audio, converts to mono 16kHz, and splits into 5-second chunks."""
    waveform, _ = librosa.load(file_path, sr=SR, mono=True)
    
    # Split into 5-second chunks
    chunks = []
    for i in range(0, len(waveform), CHUNK_SAMPLES):
        chunk = waveform[i:i + CHUNK_SAMPLES]
        # Zero-pad the last chunk if it's shorter than 5 seconds
        if len(chunk) < CHUNK_SAMPLES:
            chunk = np.pad(chunk, (0, CHUNK_SAMPLES - len(chunk)), mode='constant')
        chunks.append(chunk)
    return chunks

# --- Extraction Loop ---
for model_name, hf_repo in MODELS.items():
    print(f"\n--- Processing Model: {model_name} ---")
    model_out_dir = os.path.join(OUTPUT_DIR, model_name)
    os.makedirs(model_out_dir, exist_ok=True)
    
    # Load feature extractor and model (trust_remote_code=True for custom architectures like XEUS/BIRDAves)
    extractor = AutoFeatureExtractor.from_pretrained(hf_repo, trust_remote_code=True)
    model = AutoModel.from_pretrained(hf_repo, trust_remote_code=True).to(device)
    model.eval()
    
    all_embeddings = []
    all_labels = []
    all_filenames = []
    
    for cls_name in CLASSES:
        cls_dir = os.path.join(DATA_DIR, cls_name)
        if not os.path.isdir(cls_dir): continue
            
        print(f"Extracting for class: {cls_name}")
        for filename in tqdm(os.listdir(cls_dir)):
            if not filename.endswith('.mp3'): continue
                
            file_path = os.path.join(cls_dir, filename)
            chunks = process_audio(file_path)
            
            # Prepare chunks for the model
            inputs = extractor(chunks, sampling_rate=SR, return_tensors="pt", padding=True)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            with torch.no_grad():
                outputs = model(**inputs)
                # outputs.last_hidden_state shape: (num_chunks, sequence_length, hidden_dim)
                hidden_states = outputs.last_hidden_state
                
                # 1. Mean pool across time (sequence_length) for each chunk
                chunk_embeds = hidden_states.mean(dim=1) # Shape: (num_chunks, hidden_dim)
                
                # 2. Mean pool across all chunks to get ONE embedding for the whole audio file
                file_embed = chunk_embeds.mean(dim=0).cpu().numpy() # Shape: (hidden_dim,)
                
            all_embeddings.append(file_embed)
            all_labels.append(LABEL_MAP[cls_name])
            all_filenames.append(filename)
            
    # Save arrays to a compressed .npz file
    save_path = os.path.join(model_out_dir, "features.npz")
    np.savez_compressed(
        save_path, 
        embeddings=np.array(all_embeddings), 
        labels=np.array(all_labels), 
        filenames=np.array(all_filenames)
    )
    print(f"Saved {model_name} embeddings to {save_path}")
    
    # --- Memory Management ---
    # Crucial to prevent Kaggle GPU Out-Of-Memory errors between models
    del model, extractor, inputs, outputs, hidden_states
    torch.cuda.empty_cache()
    gc.collect()

print("\nAll extractions completed successfully!")


--- Processing Model: HuBERT ---


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Extracting for class: amecro


100%|██████████| 25/25 [00:11<00:00,  2.21it/s]


Extracting for class: amerob


...

Extracting for class: bewwre


100%|██████████| 25/25 [00:09<00:00,  2.55it/s]


Extracting for class: eastow


100%|██████████| 25/25 [00:09<00:00,  2.55it/s]


Saved WavLM embeddings to model_embeddings_eval\WavLM\features.npz

All extractions completed successfully!


## 2. AVES bio and BIRD AVES models

In [2]:
import os
import gc
import torch
import torchaudio
import librosa
import numpy as np
from tqdm import tqdm

# --- Configuration ---
DATA_DIR = "./curated_dataset/eval"  # Update if relative path is different
OUTPUT_DIR = "./model_embeddings_eval"
WEIGHTS_DIR = "./local_weights"
SR = 16000
CHUNK_LENGTH_S = 5
CHUNK_SAMPLES = SR * CHUNK_LENGTH_S

CLASSES = sorted(os.listdir(DATA_DIR))
LABEL_MAP = {cls_name: idx for idx, cls_name in enumerate(CLASSES)}

# Local model mapping to the downloaded .pt files
LOCAL_MODELS = {
    "AVES-bio": os.path.join(WEIGHTS_DIR, "aves-base-bio.torchaudio.pt"),
    "BirdAVES-biox-base": os.path.join(WEIGHTS_DIR, "birdaves-biox-base.torchaudio.pt")
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUTPUT_DIR, exist_ok=True)

def process_audio(file_path):
    """Loads audio, converts to mono 16kHz, and splits into 5-second chunks."""
    waveform, _ = librosa.load(file_path, sr=SR, mono=True)
    chunks = []
    for i in range(0, len(waveform), CHUNK_SAMPLES):
        chunk = waveform[i:i + CHUNK_SAMPLES]
        if len(chunk) < CHUNK_SAMPLES:
            chunk = np.pad(chunk, (0, CHUNK_SAMPLES - len(chunk)), mode='constant')
        chunks.append(chunk)
    return chunks

# --- Extraction Loop ---
for model_name, pt_path in LOCAL_MODELS.items():
    print(f"\n--- Processing Model: {model_name} ---")
    
    if not os.path.exists(pt_path):
        print(f"Error: Could not find {pt_path}. Please check the file path.")
        continue
        
    model_out_dir = os.path.join(OUTPUT_DIR, model_name)
    os.makedirs(model_out_dir, exist_ok=True)
    
    # 1. Load the Torchaudio HuBERT architecture and load weights
    print(f"Loading weights from {pt_path}...")
    checkpoint = torch.load(pt_path, map_location=device)
    
    if isinstance(checkpoint, torch.nn.Module):
        model = checkpoint.to(device)
    else:
        # If the checkpoint is a state_dict
        model = torchaudio.models.hubert_base()
        state_dict = checkpoint.get("model", checkpoint)
        model.load_state_dict(state_dict)
        model = model.to(device)
        
    model.eval()
    
    all_embeddings = []
    all_labels = []
    all_filenames = []
    
    # 2. Process Files
    for cls_name in CLASSES:
        cls_dir = os.path.join(DATA_DIR, cls_name)
        if not os.path.isdir(cls_dir): continue
            
        print(f"Extracting for class: {cls_name}")
        for filename in tqdm(os.listdir(cls_dir)):
            if not filename.endswith('.mp3'): continue
                
            file_path = os.path.join(cls_dir, filename)
            chunks = process_audio(file_path)
            
            with torch.no_grad():
                # Manual standardization (zero mean, unit variance) required by HuBERT
                chunks_np = np.array(chunks)
                chunks_np = (chunks_np - chunks_np.mean(axis=1, keepdims=True)) / (chunks_np.std(axis=1, keepdims=True) + 1e-7)
                
                inputs = torch.tensor(chunks_np, dtype=torch.float32).to(device)
                
                # Torchaudio's imported model returns a tuple: (features, lengths)
                features, _ = model(inputs)
                
                # Mean pool across time (dim=1), then mean pool across chunks (dim=0)
                # This condenses the 768-dimensional sequence into a single 768-D vector per file
                chunk_embeds = features.mean(dim=1) 
                file_embed = chunk_embeds.mean(dim=0).cpu().numpy() 
                
            all_embeddings.append(file_embed)
            all_labels.append(LABEL_MAP[cls_name])
            all_filenames.append(filename)
            
    # 3. Save Data
    save_path = os.path.join(model_out_dir, "features.npz")
    np.savez_compressed(
        save_path, 
        embeddings=np.array(all_embeddings), 
        labels=np.array(all_labels), 
        filenames=np.array(all_filenames)
    )
    print(f"Saved {model_name} embeddings to {save_path}")
    
    # 4. Free Memory
    del model, checkpoint, features, inputs
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

print("\nLocal extractions completed successfully!")


--- Processing Model: AVES-bio ---
Loading weights from ./local_weights\aves-base-bio.torchaudio.pt...
Extracting for class: amecro


100%|██████████| 25/25 [00:08<00:00,  2.86it/s]


Extracting for class: amerob


100%|██████████| 25/25 [00:10<00:00,  2.44it/s]


Extracting for class: astfly


...

Extracting for class: bewwre


100%|██████████| 25/25 [00:20<00:00,  1.22it/s]


Extracting for class: eastow


100%|██████████| 25/25 [00:06<00:00,  3.93it/s]


Saved BirdAVES-biox-base embeddings to ./model_embeddings_eval\BirdAVES-biox-base\features.npz

Local extractions completed successfully!


## 3. MusicHuBERT 

In [3]:
import os
import gc
import torch
import librosa
import numpy as np
from tqdm import tqdm
from transformers import AutoModel, Wav2Vec2FeatureExtractor

# --- Configuration ---
MODEL_ID = "m-a-p/MERT-v0"
DATA_DIR = "./curated_dataset/eval"
OUTPUT_DIR = "./model_embeddings_eval/MusicHuBERT"
CHUNK_LENGTH_S = 5

CLASSES = sorted(os.listdir(DATA_DIR))
LABEL_MAP = {cls_name: idx for idx, cls_name in enumerate(CLASSES)}

os.makedirs(OUTPUT_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. Load Model & Processor ---
print(f"Loading {MODEL_ID}...")
processor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True).to(device)
model.eval()

TARGET_SR = processor.sampling_rate
CHUNK_SAMPLES = TARGET_SR * CHUNK_LENGTH_S
print(f"Model sampling rate: {TARGET_SR} Hz | Chunk size: {CHUNK_SAMPLES} samples")

def process_audio(file_path):
    """Loads audio at model's native SR, converts to mono, and splits into 5s chunks."""
    waveform, _ = librosa.load(file_path, sr=TARGET_SR, mono=True)
    chunks = []
    for i in range(0, len(waveform), CHUNK_SAMPLES):
        chunk = waveform[i:i + CHUNK_SAMPLES]
        if len(chunk) < CHUNK_SAMPLES:
            chunk = np.pad(chunk, (0, CHUNK_SAMPLES - len(chunk)), mode='constant')
        chunks.append(chunk)
    return chunks

all_embeddings = []
all_labels = []
all_filenames = []

# --- 2. Extraction Loop ---
for cls_name in CLASSES:
    cls_dir = os.path.join(DATA_DIR, cls_name)
    if not os.path.isdir(cls_dir):
        continue

    print(f"\nExtracting for class: {cls_name}")
    for filename in tqdm(os.listdir(cls_dir)):
        if not filename.endswith('.mp3'):
            continue

        file_path = os.path.join(cls_dir, filename)
        chunks = process_audio(file_path)

        # Process batch of chunks through feature extractor
        inputs = processor(chunks, sampling_rate=TARGET_SR, return_tensors="pt", padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            # outputs.last_hidden_state: (batch_size, time_frames, 768)
            hidden_states = outputs.last_hidden_state

            # Mean pool across time (dim=1), then across chunks (dim=0)
            chunk_embeds = hidden_states.mean(dim=1)
            file_embed = chunk_embeds.mean(dim=0).cpu().numpy()

        all_embeddings.append(file_embed)
        all_labels.append(LABEL_MAP[cls_name])
        all_filenames.append(filename)

# --- 3. Save to NPZ Archive ---
save_path = os.path.join(OUTPUT_DIR, "features.npz")
np.savez_compressed(
    save_path,
    embeddings=np.array(all_embeddings),
    labels=np.array(all_labels),
    filenames=np.array(all_filenames)
)
print(f"\nSaved MusicHuBERT embeddings to {save_path}")

# --- 4. Free Memory ---
del model, processor, inputs, outputs, hidden_states
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

Loading m-a-p/MERT-v0...


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Model sampling rate: 16000 Hz | Chunk size: 80000 samples

Extracting for class: amecro


100%|██████████| 25/25 [00:08<00:00,  3.10it/s]



Extracting for class: amerob


...

100%|██████████| 25/25 [00:06<00:00,  3.85it/s]



Extracting for class: eastow


100%|██████████| 25/25 [00:06<00:00,  3.96it/s]



Saved MusicHuBERT embeddings to ./model_embeddings_eval/MusicHuBERT\features.npz


112

## 4.  DistilHuBERT

In [4]:
import os
import gc
import torch
import librosa
import numpy as np
from tqdm import tqdm
from transformers import HubertModel, Wav2Vec2FeatureExtractor

# --- Configuration ---
MODEL_ID = "ntu-spml/distilhubert"
DATA_DIR = "./curated_dataset/eval"
OUTPUT_DIR = "./model_embeddings_eval/DistilHuBERT"
CHUNK_LENGTH_S = 5

CLASSES = sorted(os.listdir(DATA_DIR))
LABEL_MAP = {cls_name: idx for idx, cls_name in enumerate(CLASSES)}

os.makedirs(OUTPUT_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. Load Model & Processor ---
print(f"Loading {MODEL_ID}...")
processor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_ID)
model = HubertModel.from_pretrained(MODEL_ID).to(device)
model.eval()

TARGET_SR = processor.sampling_rate  # 16000 Hz
CHUNK_SAMPLES = TARGET_SR * CHUNK_LENGTH_S

def process_audio(file_path):
    """Loads audio at 16 kHz mono and segments into 5s chunks."""
    waveform, _ = librosa.load(file_path, sr=TARGET_SR, mono=True)
    chunks = []
    for i in range(0, len(waveform), CHUNK_SAMPLES):
        chunk = waveform[i:i + CHUNK_SAMPLES]
        if len(chunk) < CHUNK_SAMPLES:
            chunk = np.pad(chunk, (0, CHUNK_SAMPLES - len(chunk)), mode='constant')
        chunks.append(chunk)
    return chunks

all_embeddings = []
all_labels = []
all_filenames = []

# --- 2. Feature Extraction Loop ---
for cls_name in CLASSES:
    cls_dir = os.path.join(DATA_DIR, cls_name)
    if not os.path.isdir(cls_dir):
        continue

    print(f"\nExtracting for class: {cls_name}")
    for filename in tqdm(os.listdir(cls_dir)):
        if not filename.endswith('.mp3'):
            continue

        file_path = os.path.join(cls_dir, filename)
        chunks = process_audio(file_path)

        # The Wav2Vec2FeatureExtractor handles standardisation automatically
        inputs = processor(chunks, sampling_rate=TARGET_SR, return_tensors="pt", padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            # last_hidden_state shape: (batch_size, time_frames, 768)
            hidden_states = outputs.last_hidden_state

            # Mean pool across time sequence (dim=1), then average across 5s chunks (dim=0)
            chunk_embeds = hidden_states.mean(dim=1)
            file_embed = chunk_embeds.mean(dim=0).cpu().numpy()

        all_embeddings.append(file_embed)
        all_labels.append(LABEL_MAP[cls_name])
        all_filenames.append(filename)

# --- 3. Save to NPZ Archive ---
save_path = os.path.join(OUTPUT_DIR, "features.npz")
np.savez_compressed(
    save_path,
    embeddings=np.array(all_embeddings),
    labels=np.array(all_labels),
    filenames=np.array(all_filenames)
)
print(f"\nSaved DistilHuBERT embeddings to {save_path}")

# --- 4. Clean Memory ---
del model, processor, inputs, outputs, hidden_states
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

Loading ntu-spml/distilhubert...


Loading weights:   0%|          | 0/49 [00:00<?, ?it/s]


Extracting for class: amecro


100%|██████████| 25/25 [00:04<00:00,  5.99it/s]



Extracting for class: amerob


...

100%|██████████| 25/25 [00:04<00:00,  5.06it/s]



Extracting for class: eastow


100%|██████████| 25/25 [00:05<00:00,  4.74it/s]



Saved DistilHuBERT embeddings to ./model_embeddings_eval/DistilHuBERT\features.npz


109

# Embedding Storage Specification (`features.npz`)

Audio embeddings extracted from SSL models (HuBERT, IndicHuBERT, MusicHuBERT, XEUS) are serialized into compressed NumPy archive files (`.npz`) using `np.savez_compressed()`.

---

## 1. Internal Archive Schema

Each `.npz` archive contains three parallel, synchronized arrays accessible as dictionary keys:

| Key | Type | Dimensions | Description |
| :--- | :--- | :--- | :--- |
| **`embeddings`** | `np.float32` | `(N, D)` | 2D matrix of file-level feature representations.<br>• $N$: Total number of processed audio tracks.<br>• $D$: Feature hidden dimension (**768** for HuBERT-Base/IndicHuBERT, **1024** for XEUS). |
| **`labels`** | `np.int64` | `(N,)` | 1D vector of integer class labels assigned alphabetically from directory names (`0` to $C-1$). |
| **`filenames`** | `np.str_` | `(N,)` | 1D vector of original file basenames (e.g., `sample_001.mp3`) used for sample tracking and audit splits. |

---

## 2. Row Alignment Guarantee

All three arrays share the exact same row indexing $i \in [0, N-1]$:

$$i\text{-th Entry} \iff \begin{cases} \mathbf{x}_i = \texttt{embeddings}[i] & \in \mathbb{R}^D \\  y_i = \texttt{labels}[i] & \in \{0, \dots, C-1\} \\  s_i = \texttt{filenames}[i] & \text{(Source file name)} \end{cases}$$

---

## 3. Loading and Inspection in Python

```python
import numpy as np

# Load the archive
archive_path = "./model_embeddings/IndicHuBERT/features.npz"
data = np.load(archive_path)

# Extract stored components
X = data["embeddings"]
y = data["labels"]
filenames = data["filenames"]

print("Archived keys:", data.files)
print(f"Embeddings shape : {X.shape} (dtype: {X.dtype})")
print(f"Labels shape     : {y.shape} (dtype: {y.dtype})")
print(f"Filenames shape  : {filenames.shape} (dtype: {filenames.dtype})")